# Soil eDNA Metabarcoding Pipeline Analysis (PR2 Database)
### A Reproducible Workflow for MinION Amplicon Sequencing (JEDI & COI)
**Project:** Genorobotics Semester Project (EPFL)

**Markers:** JEDI (~460 bp COI, arthropod-optimized) & Standard COI (~658 bp Folmer)

**Database:** PR2 PR2 — a curated eukaryotic COI database covering Metazoa, Fungi, Protists, and Plants

---

## 0. Critical Review: PR2 vs Porter CO1

### Why PR2?
The previous analysis used Porter CO1, a general-purpose COI database that is heavily biased toward Metazoa (mostly Arthropoda and Chordata). PR2 is built on the PR2 taxonomy and includes broader eukaryotic coverage:
* **Protists & Diatoms:** PR2 includes Stramenopiles, Alveolata, and other non-metazoan eukaryotes that Porter CO1 only treated as outgroups.
* **Fungi:** Better representation of soil fungi (Basidiomycota, Ascomycota).
* **Clean taxonomy:** PR2-based 9-level taxonomy without NCBI taxon ID artifacts.
* **Smaller database (15,947 sequences):** Much faster SINTAX queries compared to Porter CO1 (9.4GB). However, this also means less species-level resolution for metazoans.

### PR2 Taxonomy Mapping (PR2 → SINTAX)
| SINTAX level | PR2 field | Example |
|---|---|---|
| `d:` Domain | Kingdom | Eukaryota |
| `k:` Kingdom | Supergroup | Obazoa, TSAR, Archaeplastida |
| `p:` Phylum | Division | Opisthokonta, Alveolata |
| `c:` Class | Subdivision | Metazoa, Fungi, Apicomplexa |
| `o:` Order | Class | Tardigrada, Nemertea |
| `f:` Family | Family | resolved from Order/Family fields |
| `g:` Genus | Genus | Milnesium |
| `s:` Species | Species | Milnesium_sp. |

### Limitations
* **15,947 sequences** vs Porter CO1's 2.2M — far fewer reference sequences, especially for metazoans.
* Species-level resolution may be lower for common arthropods/chordates.
* The PR2 taxonomy hierarchy differs from NCBI — "Phylum" in the output corresponds to PR2's Division level (e.g., Opisthokonta, Stramenopiles).

### 1. The JEDI Advantage: Optimized for Soil Arthropods
The JEDI primers target a ~460 bp region of the COI gene, specifically designed for arthropod detection in environmental samples.
* **Expected Dominance:** Arthropoda (insects, collembolans, arachnids), Annelida (earthworms), and Nematoda should dominate.
* **Potential Limitation:** The shorter fragment may reduce species-level resolution compared to full-length COI.

### 2. Standard COI in Soil: Expected Challenges
The standard Folmer COI primers (~658 bp) may underperform in soil matrices:
* **DNA Degradation:** Soil humic acids degrade DNA, favoring shorter amplicons (JEDI) over longer ones (COI).
* **Primer Bias:** Folmer primers were designed for a broader metazoan range but may miss soil-specific taxa that JEDI captures.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
from pathlib import Path
from Bio import SeqIO
from matplotlib import cm

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 12,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10
})

# Define a function to clean sample names (e.g., "Sample_barcode01" -> "01")
def clean_sample_names(columns):
    return [c.replace('Sample_barcode', '').replace('Sample_', '') for c in columns]

# Auto-detect taxonomy column prefix (SILVA_, MIDORI_, or PR2_)
def get_tax_prefix(df):
    for col in df.columns:
        if col.startswith("SILVA_"):
            return "SILVA"
        if col.startswith("Porter_"):
            return "Porter CO1"
        if col.startswith("MIDORI_"):
            return "MIDORI"
        if col.startswith("PR2_"):
            return "PR2"
    return "SILVA"

BASE = Path("out/Soil_eDNA_JEDI_COI_14_01_26")

# Global confidence threshold for taxonomy filtering
CONF_THRESHOLD = 0.8

def stacked_bar_compare(df, rank, prefix, marker_label, sample_cols, top_n=10, conf_threshold=None):
    """Two side-by-side stacked bars: left includes Unassigned, right excludes it.

    - Blanks low-confidence assignments (< conf_threshold) before grouping.
    - Keeps Unassigned in the data so it\'s visible in the left panel with
      a dedicated gray color + legend entry.
    - Right panel drops Unassigned and renormalizes to 100% of assigned reads,
      useful when Unassigned dominates (e.g. soil COI).
    """
    if conf_threshold is None:
        conf_threshold = CONF_THRESHOLD
    col = f'{prefix}_{rank}'
    conf_col = f'{prefix}_{rank}_Conf'
    d = df.copy()
    if conf_col in d.columns:
        d.loc[pd.to_numeric(d[conf_col], errors='coerce').fillna(0) < conf_threshold, col] = ''
    d[col] = d[col].replace('', 'Unassigned').fillna('Unassigned')
    grouped = d.groupby(col)[sample_cols].sum()

    unassigned_row = grouped.loc['Unassigned'] if 'Unassigned' in grouped.index else None
    assigned = grouped.drop('Unassigned', errors='ignore').copy()
    assigned['Total'] = assigned.sum(axis=1)
    assigned = assigned.sort_values('Total', ascending=False)
    top_idx = assigned.head(top_n).index
    top_data = assigned.loc[top_idx].drop(columns='Total')
    others_data = assigned.loc[~assigned.index.isin(top_idx)].drop(columns='Total').sum()

    # Build LEFT (with Unassigned) and RIGHT (without)
    left = top_data.copy()
    left.loc['Others'] = others_data
    if unassigned_row is not None:
        left.loc['Unassigned'] = unassigned_row
    left_pct = left.div(left.sum(axis=0).replace(0, np.nan), axis=1) * 100
    left_pct = left_pct.fillna(0)

    right = top_data.copy()
    right.loc['Others'] = others_data
    right_pct = right.div(right.sum(axis=0).replace(0, np.nan), axis=1) * 100
    right_pct = right_pct.fillna(0)

    def colors_for(data_):
        palette = cm.tab20(np.linspace(0, 1, max(len(data_) - 2, 1)))
        out = []
        pi = 0
        for name in data_.index:
            if name == 'Unassigned':
                out.append('#D3D3D3')
            elif name == 'Others':
                out.append('#696969')
            else:
                out.append(palette[pi % len(palette)])
                pi += 1
        return out

    # Report Unassigned share for context
    if unassigned_row is not None:
        total_reads = grouped.sum().sum()
        un_share = 100 * unassigned_row.sum() / total_reads if total_reads else 0
        print(f'[{marker_label}] {rank}: Unassigned = {un_share:.1f}% of reads')

    fig, axes = plt.subplots(1, 2, figsize=(20, 7), sharey=True)
    for ax, data, subtitle in [(axes[0], left_pct, 'including Unassigned'),
                                (axes[1], right_pct, 'excluding Unassigned (renormalized)')]:
        data = data.copy()
        data.columns = clean_sample_names(data.columns)
        data.T.plot(kind='bar', stacked=True, ax=ax, width=0.85, color=colors_for(data))
        ax.set_title(f'{rank}-Level Composition ({marker_label}, {prefix})\n{subtitle}',
                     fontweight='bold', fontsize=11)
        ax.set_ylabel('Relative Abundance (%)')
        ax.set_xlabel('Sample')
        h, l = ax.get_legend_handles_labels()
        ax.legend(reversed(h), reversed(l), bbox_to_anchor=(1.02, 1), loc='upper left',
                  title=rank, fontsize=8)
        for _i, _s in enumerate(data.columns):
            if data[_s].sum() == 0:
                ax.text(_i, 2, 'ND', ha='center', va='bottom', fontsize=9,
                        color='gray', fontstyle='italic')
        ax.set_ylim(0, 105)
    plt.tight_layout()
    add_conf_note(fig, 'filtered')
    plt.show()

# Confidence-interpretation caption helper
_CONF_NOTES = {
    'filtered':   'SINTAX confidence \u2265 0.8 applied \u2014 lower-confidence calls treated as Unassigned.',
    'unfiltered': 'No confidence filter applied \u2014 includes low-confidence assignments. Interpret taxa cautiously.',
    'blast':      'Based on BLAST vs NCBI identity (not SINTAX confidence).',
    'qc':         'Pre-taxonomy QC plot \u2014 confidence filter not applicable.',
    'meta':       'Pipeline metadata \u2014 confidence filter not applicable.',
}
def add_conf_note(fig=None, kind='filtered'):
    if fig is None:
        fig = plt.gcf()
    txt = _CONF_NOTES.get(kind, _CONF_NOTES['filtered'])
    fig.text(0.5, -0.02, txt, ha='center', va='top', fontsize=9,
             style='italic', color='#555555', wrap=True)


## Table Headers & Data Structure
Inspect the comprehensive taxonomy CSV files to verify column names and data shape.

In [ ]:
# Load both datasets
df_jedi_raw = pd.read_csv(BASE / 'taxonomy_summary/pr2-porter/comprehensive_taxonomy_JEDI.csv')
df_coi_raw = pd.read_csv(BASE / 'taxonomy_summary/pr2-porter/comprehensive_taxonomy_COI.csv')

prefix_jedi = get_tax_prefix(df_jedi_raw)
prefix_coi = get_tax_prefix(df_coi_raw)

for label, df, pfx in [("JEDI", df_jedi_raw, prefix_jedi), ("COI", df_coi_raw, prefix_coi)]:
    print(f"\n{'='*60}")
    print(f"  {label} ({pfx} database)  \u2014  {df.shape[0]} OTUs \u00d7 {df.shape[1]} columns")
    print(f"{'='*60}")
    
    sample_cols = [c for c in df.columns if c.startswith("Sample_")]
    taxonomy_cols = [c for c in df.columns if c.startswith(f"{pfx}_") or c.startswith("NCBI_")]
    meta_cols = [c for c in df.columns if c not in sample_cols + taxonomy_cols]
    
    print(f"\nMetadata columns:  {meta_cols}")
    print(f"Sample columns:    {sample_cols}")
    print(f"Taxonomy columns:  {taxonomy_cols}")
    print(f"\nFirst 3 rows:")
    display(df.head(3))

## Confidence Distribution Dashboard

In [ ]:
# === SINTAX Confidence Distribution Dashboard ===
ranks = ['Phylum', 'Class', 'Order', 'Family', 'Genus', 'Species']

def plot_confidence_row(df, prefix, marker_label, axes):
    """Plot confidence metrics for one marker."""
    conf_cols = [f'{prefix}_{r}_Conf' for r in ranks]
    existing = [c for c in conf_cols if c in df.columns]
    if not existing:
        for ax in axes:
            ax.text(0.5, 0.5, 'No confidence columns found.\nRegenerate with:\nbash regenerate_taxonomy.sh',
                    ha='center', va='center', transform=ax.transAxes, fontsize=10, color='red')
            ax.set_title(marker_label)
        return

    means, counts = [], []
    for r in ranks:
        col = f'{prefix}_{r}_Conf'
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors='coerce').dropna()
            means.append(vals.mean() if len(vals) > 0 else 0)
            counts.append(len(vals))
        else:
            means.append(0)
            counts.append(0)

    ax1 = axes[0]
    colors = ['#2ecc71' if m >= 0.8 else '#e74c3c' for m in means]
    bars = ax1.bar(ranks, means, color=colors, edgecolor='white')
    ax1.axhline(y=0.8, color='red', linestyle='--', alpha=0.7, label='Threshold (0.8)')
    ax1.set_ylim(0, 1.05)
    ax1.set_ylabel('Mean Confidence')
    ax1.set_title(f'{marker_label}: Mean Confidence per Rank', fontweight='bold')
    ax1.legend(fontsize=8)
    ax1.tick_params(axis='x', rotation=45)
    for bar, m, n in zip(bars, means, counts):
        if m > 0:
            ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{m:.2f} (n={n})', ha='center', va='bottom', fontsize=7)

    hist_rank = 'Genus' if prefix == 'SILVA' else 'Phylum'
    hist_col = f'{prefix}_{hist_rank}_Conf'
    ax2 = axes[1]
    if hist_col in df.columns:
        vals = pd.to_numeric(df[hist_col], errors='coerce').dropna()
        if len(vals) > 0:
            ax2.hist(vals, bins=20, range=(0, 1), color='steelblue', edgecolor='white', alpha=0.8)
            ax2.axvline(x=0.8, color='red', linestyle='--', linewidth=2, label='Threshold (0.8)')
            ax2.set_xlabel('Confidence')
            ax2.set_ylabel('OTU Count')
            ax2.set_title(f'{marker_label}: {hist_rank}-Level Confidence Distribution', fontweight='bold')
            ax2.legend(fontsize=8)
        else:
            ax2.text(0.5, 0.5, f'No {hist_rank} confidence values',
                    ha='center', va='center', transform=ax2.transAxes)
            ax2.set_title(f'{marker_label}: {hist_rank}-Level Confidence', fontweight='bold')
    else:
        ax2.text(0.5, 0.5, f'Column {hist_col} not found',
                ha='center', va='center', transform=ax2.transAxes)
        ax2.set_title(f'{marker_label}: {hist_rank}-Level Confidence', fontweight='bold')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('SINTAX Confidence Distribution Dashboard', fontsize=16, fontweight='bold', y=1.02)

plot_confidence_row(df_jedi_raw, prefix_jedi, f'JEDI ({prefix_jedi})', axes[0])
plot_confidence_row(df_coi_raw, prefix_coi, f'COI ({prefix_coi})', axes[1])

plt.tight_layout()
add_conf_note(kind='filtered')
plt.show()

## Database Performance Dashboard

Evaluate how well the reference database classifies our OTUs. Three diagnostic panels per marker:

1. **OTUs classified per rank** — What fraction of OTUs gets any assignment (gray) vs a *confident* assignment (green, ≥ 0.8) at each rank. Shows **coverage breadth + quality**.
2. **Reads covered per rank** — Same but weighted by read abundance. A DB could classify few OTUs that nonetheless represent most reads (or the opposite).
3. **Deepest confident rank** — For each OTU, the deepest rank it reached confidently. Shows the **resolution power** of the DB for this dataset.

Good DB for this marker = high green bars + reads-covered high + most OTUs reaching Genus/Species.

In [ ]:
# Database Performance Dashboard
# Shows how well the reference DB classifies our OTUs: assignment rate per rank,
# confident-assignment rate, read-weighted coverage, and resolution depth.

ranks = ['Phylum', 'Class', 'Order', 'Family', 'Genus', 'Species']
marker_data = [(df_jedi_raw, prefix_jedi, 'JEDI'), (df_coi_raw, prefix_coi, 'COI')]

def db_stats(df, prefix):
    """Return per-rank stats for a marker."""
    stats = {'rank': [], 'pct_any': [], 'pct_conf': [],
             'pct_reads_any': [], 'pct_reads_conf': []}
    total_otus = len(df)
    total_reads = df['Total_Abundance'].sum() if 'Total_Abundance' in df.columns else df[[c for c in df.columns if c.startswith('Sample_')]].sum().sum()
    for rank in ranks:
        col = f'{prefix}_{rank}'
        conf_col = f'{prefix}_{rank}_Conf'
        if col not in df.columns:
            continue
        has_any = ~df[col].isin(['Unassigned', '', None]) & df[col].notna()
        if conf_col in df.columns:
            conf_vals = pd.to_numeric(df[conf_col], errors='coerce').fillna(0)
            has_conf = has_any & (conf_vals >= CONF_THRESHOLD)
        else:
            has_conf = has_any
        reads_col = 'Total_Abundance' if 'Total_Abundance' in df.columns else None
        if reads_col:
            reads_any = df.loc[has_any, reads_col].sum()
            reads_conf = df.loc[has_conf, reads_col].sum()
        else:
            sample_cols = [c for c in df.columns if c.startswith('Sample_')]
            reads_any = df.loc[has_any, sample_cols].sum().sum()
            reads_conf = df.loc[has_conf, sample_cols].sum().sum()
        stats['rank'].append(rank)
        stats['pct_any'].append(100 * has_any.sum() / total_otus)
        stats['pct_conf'].append(100 * has_conf.sum() / total_otus)
        stats['pct_reads_any'].append(100 * reads_any / total_reads if total_reads > 0 else 0)
        stats['pct_reads_conf'].append(100 * reads_conf / total_reads if total_reads > 0 else 0)
    return stats

def resolution_depth(df, prefix):
    """For each OTU, find deepest confident rank. Returns counts per rank + Unassigned."""
    depths = {'Unassigned': 0}
    for r in ranks:
        depths[r] = 0
    for _, row in df.iterrows():
        deepest = 'Unassigned'
        for rank in ranks:
            col = f'{prefix}_{rank}'
            conf_col = f'{prefix}_{rank}_Conf'
            if col not in df.columns:
                continue
            val = row.get(col, '')
            if pd.isna(val) or val in ('Unassigned', ''):
                continue
            if conf_col in df.columns:
                try:
                    if float(row.get(conf_col, 0) or 0) >= CONF_THRESHOLD:
                        deepest = rank
                except (ValueError, TypeError):
                    pass
            else:
                deepest = rank
        depths[deepest] += 1
    return depths

# Build figure: one row per marker, 3 subplots per row
n_markers = len(marker_data)
fig, axes = plt.subplots(n_markers, 3, figsize=(18, 5 * n_markers))
if n_markers == 1:
    axes = axes.reshape(1, -1)

fig.suptitle('Database Performance Dashboard', fontsize=16, fontweight='bold', y=1.00)

for row_idx, (df, prefix, label) in enumerate(marker_data):
    s = db_stats(df, prefix)
    x = np.arange(len(s['rank']))
    width = 0.4

    # --- Panel 1: OTU-based assignment rate ---
    ax = axes[row_idx, 0]
    ax.bar(x - width/2, s['pct_any'], width, color='#95a5a6', label='Any assignment')
    ax.bar(x + width/2, s['pct_conf'], width, color='#2d8a4e', label=f'Confident (\u2265 {CONF_THRESHOLD})')
    ax.set_xticks(x)
    ax.set_xticklabels(s['rank'], rotation=45)
    ax.set_ylabel('% of OTUs')
    ax.set_ylim(0, 105)
    ax.set_title(f'{label}: OTUs classified per rank', fontweight='bold', fontsize=11)
    ax.legend(fontsize=8)
    for xi, (a, c) in enumerate(zip(s['pct_any'], s['pct_conf'])):
        ax.text(xi - width/2, a + 1, f'{a:.0f}%', ha='center', fontsize=7)
        ax.text(xi + width/2, c + 1, f'{c:.0f}%', ha='center', fontsize=7, color='#2d8a4e')

    # --- Panel 2: Read-weighted assignment rate ---
    ax = axes[row_idx, 1]
    ax.bar(x - width/2, s['pct_reads_any'], width, color='#95a5a6', label='Any assignment')
    ax.bar(x + width/2, s['pct_reads_conf'], width, color='#2d8a4e', label=f'Confident (\u2265 {CONF_THRESHOLD})')
    ax.set_xticks(x)
    ax.set_xticklabels(s['rank'], rotation=45)
    ax.set_ylabel('% of reads')
    ax.set_ylim(0, 105)
    ax.set_title(f'{label}: Reads covered per rank', fontweight='bold', fontsize=11)
    ax.legend(fontsize=8)

    # --- Panel 3: Resolution depth ---
    ax = axes[row_idx, 2]
    depths = resolution_depth(df, prefix)
    labels_d = ['Unassigned'] + ranks
    counts_d = [depths.get(l, 0) for l in labels_d]
    total_d = sum(counts_d)
    pcts_d = [100 * c / total_d if total_d > 0 else 0 for c in counts_d]
    colors_d = ['#c0392b'] + ['#e6a817', '#f39c12', '#27ae60', '#16a085', '#2d8a4e', '#1a5f37']
    bars = ax.bar(labels_d, pcts_d, color=colors_d)
    ax.set_ylabel('% of OTUs')
    ax.set_ylim(0, max(pcts_d) * 1.15 if pcts_d else 100)
    ax.set_title(f'{label}: Deepest confident rank', fontweight='bold', fontsize=11)
    ax.tick_params(axis='x', rotation=45)
    for xi, (p, c) in enumerate(zip(pcts_d, counts_d)):
        ax.text(xi, p + 0.5, f'{p:.0f}%\n(n={c})', ha='center', fontsize=7)

plt.tight_layout()
add_conf_note(kind='filtered')
plt.show()

# Print summary table
print("\n" + "=" * 70)
print("DATABASE PERFORMANCE SUMMARY")
print("=" * 70)
for df, prefix, label in marker_data:
    s = db_stats(df, prefix)
    print(f"\n{label}  (DB: {prefix})")
    print(f"  {'Rank':<10} {'% OTUs any':>12} {'% OTUs conf':>14} {'% reads conf':>14}")
    for i, r in enumerate(s['rank']):
        print(f"  {r:<10} {s['pct_any'][i]:>11.1f}% {s['pct_conf'][i]:>13.1f}% {s['pct_reads_conf'][i]:>13.1f}%")


## Raw Read Length Distributions (Pre-Clustering)
Number of reads at each sequence length, aggregated across all barcodes for each marker.

In [ ]:
import gzip

barcode_dirs = sorted(BASE.glob("barcode*"))
marker_lengths = {"JEDI": [], "COI": []}
marker_colors_raw = {"JEDI": "#9C27B0", "COI": "#FF9800"}

for bd in barcode_dirs:
    for marker in marker_lengths:
        fq = bd / f"filtered_reads_{marker}.fastq.gz"
        if fq.exists():
            with gzip.open(str(fq), 'rt') as f:
                for i, line in enumerate(f):
                    if i % 4 == 1:
                        marker_lengths[marker].append(len(line.strip()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, (marker, lengths) in enumerate(marker_lengths.items()):
    ax = axes[i]
    if lengths:
        ax.hist(lengths, bins=100, color=marker_colors_raw[marker], edgecolor="white", alpha=0.85)
        ax.axvline(np.median(lengths), color="red", ls="--", lw=1.5,
                   label=f"median={np.median(lengths):.0f}bp")
        ax.set_title(f"{marker} \u2014 {len(lengths):,} reads", fontsize=13, fontweight="bold")
        ax.set_xlabel("Read length (bp)")
        ax.set_ylabel("Number of reads")
        ax.legend()
        print(f"\u2713 {marker}: {len(lengths):,} reads, median={np.median(lengths):.0f}bp, "
              f"range={min(lengths)}-{max(lengths)}bp")
    else:
        ax.text(0.5, 0.5, f"{marker}\nNo reads found", ha='center', va='center', transform=ax.transAxes)

plt.suptitle("Raw Read Length Distributions (Soil)", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
add_conf_note(kind='qc')
plt.show()

## Consensus OTU Sequence Length Distributions
Length distributions of consensus OTU sequences for each marker.

In [ ]:
fasta_files = {
    "JEDI": BASE / "temp_clustering/consensus_JEDI_clean.fasta",
    "COI":  BASE / "temp_clustering/consensus_COI_clean.fasta",
}
marker_colors = {"JEDI": "#9C27B0", "COI": "#FF9800"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, (marker, path) in enumerate(fasta_files.items()):
    ax = axes[i]
    if path.exists():
        lengths = [len(rec.seq) for rec in SeqIO.parse(str(path), "fasta")]
        ax.hist(lengths, bins=50, color=marker_colors[marker], edgecolor="white", alpha=0.85)
        ax.axvline(np.median(lengths), color="red", ls="--", lw=1.5,
                   label=f"median={np.median(lengths):.0f}bp")
        ax.set_title(f"{marker} \u2014 {len(lengths)} OTUs", fontsize=13, fontweight="bold")
        ax.set_xlabel("Sequence length (bp)")
        ax.set_ylabel("Number of OTUs")
        ax.legend()
        print(f"\u2713 {marker}: {len(lengths)} OTUs, median={np.median(lengths):.0f}bp, "
              f"range={min(lengths)}-{max(lengths)}bp")
    else:
        ax.text(0.5, 0.5, f"{marker}\nFASTA not found", ha='center', va='center', transform=ax.transAxes)

plt.suptitle("Consensus OTU Sequence Length Distributions (Soil)", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
add_conf_note(kind='qc')
plt.show()

---
# Part A: JEDI Marker Biodiversity Analysis (PR2)
*Objective: Characterize the soil invertebrate community using the JEDI marker (~460 bp COI) with the PR2 PR2 reference database.*

## A.1a Broad Taxonomic Structure — Phylum
Note: In the PR2/PR2 taxonomy, "Phylum" corresponds to the PR2 Division level (e.g., Opisthokonta, Stramenopiles, Alveolata).
* **Expected:** Opisthokonta should dominate (containing Metazoa), with possible Stramenopiles (oomycetes/diatoms) and Alveolata signals.

In [ ]:
df_jedi = df_jedi_raw.copy() if "df_jedi" not in dir() else df_jedi
sample_cols_jedi = [c for c in df_jedi.columns if c.startswith('Sample_') and 'unclassified' not in c]
stacked_bar_compare(df_jedi, 'Phylum', prefix_jedi, 'JEDI', sample_cols_jedi, top_n=10)


## A.1b Class-Level Breakdown (JEDI)
In PR2/PR2 taxonomy, "Class" corresponds to the Subdivision level (e.g., Metazoa, Fungi, Apicomplexa).
* **Metazoa** should dominate as the primary soil eukaryote signal.
* Look for **Fungi** signals that Porter CO1 would have missed.

In [ ]:
sample_cols_jedi = [c for c in df_jedi.columns if c.startswith('Sample_') and 'unclassified' not in c]
stacked_bar_compare(df_jedi, 'Class', prefix_jedi, 'JEDI', sample_cols_jedi, top_n=15)


## A.2 Order-Level Breakdown (JEDI)
In PR2/PR2, "Order" maps to the PR2 Class level (e.g., Tardigrada, Insecta, Arachnida for Metazoa).

In [ ]:
sample_cols_jedi = [c for c in df_jedi.columns if c.startswith('Sample_') and 'unclassified' not in c]
stacked_bar_compare(df_jedi, 'Order', prefix_jedi, 'JEDI', sample_cols_jedi, top_n=15)


## A.3 Genus-Level Top 20 (JEDI)
Top genera detected by the JEDI marker with PR2 taxonomy.

In [ ]:
# Top 20 Genera — Confident assignments only (confidence >= 0.8)
genus_col = f'{prefix_jedi}_Genus'
conf_col = f'{prefix_jedi}_Genus_Conf'
df_jedi[genus_col] = df_jedi[genus_col].fillna('Unassigned')

# Filter to only OTUs with genus confidence >= 0.8
if conf_col in df_jedi.columns:
    mask_conf = pd.to_numeric(df_jedi[conf_col], errors='coerce') >= 0.8
    df_conf = df_jedi[mask_conf].copy()
else:
    df_conf = df_jedi[df_jedi[genus_col] != 'Unassigned'].copy()

genus_agg = df_conf.groupby(genus_col)[sample_cols_jedi].sum()
genus_agg = genus_agg.drop('Unassigned', errors='ignore')
genus_agg['Total'] = genus_agg.sum(axis=1)
genus_agg = genus_agg.sort_values('Total', ascending=False)
top20 = genus_agg.head(20)

# Mean confidence per genus
genus_conf = {}
if conf_col in df_conf.columns:
    for genus in top20.index:
        mask = df_conf[genus_col] == genus
        vals = pd.to_numeric(df_conf.loc[mask, conf_col], errors='coerce').dropna()
        genus_conf[genus] = vals.mean() if len(vals) > 0 else None

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(x=top20['Total'], y=top20.index, palette='viridis', ax=ax)
ax.set_title(f'Top 20 Genera — Confident Only (\u2265 0.8) (JEDI, {prefix_jedi})', fontweight='bold')
ax.set_xlabel('Total Abundance')
ax.set_ylabel('Genus')

if genus_conf:
    for j, genus in enumerate(top20.index):
        conf = genus_conf.get(genus)
        if conf is not None:
            ax.text(top20.loc[genus, 'Total'] + 0.001, j,
                    f' ({conf:.2f})', va='center', fontsize=9, color='darkred', fontweight='bold')

plt.tight_layout()
add_conf_note(kind='filtered')
plt.show()

### Top 20 Genera by Abundance — All Confidences (JEDI)
Same marker but showing ALL genus assignments regardless of confidence, ranked by abundance. Bars are color-coded by mean confidence level.

In [ ]:
# Top 20 Genera by ABUNDANCE — All confidence levels
genus_col = f'{prefix_jedi}_Genus'
conf_col = f'{prefix_jedi}_Genus_Conf'
df_tmp = df_jedi.copy()
df_tmp[genus_col] = df_tmp[genus_col].fillna('Unassigned')

genus_abund = df_tmp.groupby(genus_col)[sample_cols_jedi].sum()
genus_abund = genus_abund.drop('Unassigned', errors='ignore')
genus_abund['Total'] = genus_abund.sum(axis=1)
genus_abund = genus_abund.sort_values('Total', ascending=False)
top20 = genus_abund.head(20)

# Mean confidence per genus
raw_conf = {}
if conf_col in df_tmp.columns:
    for genus in top20.index:
        mask = df_tmp[genus_col] == genus
        vals = pd.to_numeric(df_tmp.loc[mask, conf_col], errors='coerce').dropna()
        raw_conf[genus] = vals.mean() if len(vals) > 0 else None

fig, ax = plt.subplots(figsize=(11, 8))
colors = []
for genus in top20.index:
    c = raw_conf.get(genus)
    if c is None:
        colors.append('#999999')
    elif c >= 0.8:
        colors.append('#2d8a4e')
    elif c >= 0.5:
        colors.append('#e6a817')
    else:
        colors.append('#c0392b')

bars = ax.barh(range(len(top20)), top20['Total'].values[::-1], color=colors[::-1], height=0.7)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20.index[::-1], fontsize=9)
ax.set_xlabel('Total Abundance', fontsize=11)
ax.set_title(f'Top 20 Genera by Abundance — All Confidences (JEDI, {prefix_jedi})',
             fontweight='bold', fontsize=12)

xmax = top20['Total'].max()
for j, genus in enumerate(top20.index[::-1]):
    c = raw_conf.get(genus)
    if c is not None:
        color = '#2d8a4e' if c >= 0.8 else '#e6a817' if c >= 0.5 else '#c0392b'
        ax.text(top20.loc[genus, 'Total'] + xmax * 0.01, j,
                f' {c:.2f}', va='center', fontsize=9, fontweight='bold', color=color)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2d8a4e', label='\u2265 0.80 (confident)'),
    Patch(facecolor='#e6a817', label='0.50 \u2013 0.79 (moderate)'),
    Patch(facecolor='#c0392b', label='< 0.50 (low confidence)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=8,
          title='Mean confidence', title_fontsize=9, framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
add_conf_note(kind='unfiltered')
plt.show()

---
# Part B: COI Marker Biodiversity Analysis (Standard Folmer ~658 bp, PR2)
*Objective: Characterize the soil community using the standard COI marker with PR2 taxonomy and compare with JEDI.*

## B.1a Broad Taxonomic Structure (COI)

In [ ]:
df_coi = df_coi_raw.copy() if "df_coi" not in dir() else df_coi
sample_cols_coi = [c for c in df_coi.columns if c.startswith('Sample_') and 'unclassified' not in c]
stacked_bar_compare(df_coi, 'Phylum', prefix_coi, 'COI', sample_cols_coi, top_n=10)


## B.1b Class-Level Breakdown (COI)

In [ ]:
sample_cols_coi = [c for c in df_coi.columns if c.startswith('Sample_') and 'unclassified' not in c]
stacked_bar_compare(df_coi, 'Class', prefix_coi, 'COI', sample_cols_coi, top_n=15)


## B.2 Order-Level Breakdown (COI)

In [ ]:
sample_cols_coi = [c for c in df_coi.columns if c.startswith('Sample_') and 'unclassified' not in c]
stacked_bar_compare(df_coi, 'Order', prefix_coi, 'COI', sample_cols_coi, top_n=15)


## B.3 Genus-Level Top 20 (COI)

In [ ]:
# Top 20 Genera — Confident assignments only (confidence >= 0.8)
genus_col = f'{prefix_coi}_Genus'
conf_col = f'{prefix_coi}_Genus_Conf'
df_coi[genus_col] = df_coi[genus_col].fillna('Unassigned')

# Filter to only OTUs with genus confidence >= 0.8
if conf_col in df_coi.columns:
    mask_conf = pd.to_numeric(df_coi[conf_col], errors='coerce') >= 0.8
    df_conf = df_coi[mask_conf].copy()
else:
    df_conf = df_coi[df_coi[genus_col] != 'Unassigned'].copy()

genus_agg = df_conf.groupby(genus_col)[sample_cols_coi].sum()
genus_agg = genus_agg.drop('Unassigned', errors='ignore')
genus_agg['Total'] = genus_agg.sum(axis=1)
genus_agg = genus_agg.sort_values('Total', ascending=False)
top20 = genus_agg.head(20)

# Mean confidence per genus
genus_conf = {}
if conf_col in df_conf.columns:
    for genus in top20.index:
        mask = df_conf[genus_col] == genus
        vals = pd.to_numeric(df_conf.loc[mask, conf_col], errors='coerce').dropna()
        genus_conf[genus] = vals.mean() if len(vals) > 0 else None

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(x=top20['Total'], y=top20.index, palette='viridis', ax=ax)
ax.set_title(f'Top 20 Genera — Confident Only (\u2265 0.8) (COI, {prefix_coi})', fontweight='bold')
ax.set_xlabel('Total Abundance')
ax.set_ylabel('Genus')

if genus_conf:
    for j, genus in enumerate(top20.index):
        conf = genus_conf.get(genus)
        if conf is not None:
            ax.text(top20.loc[genus, 'Total'] + 0.001, j,
                    f' ({conf:.2f})', va='center', fontsize=9, color='darkred', fontweight='bold')

plt.tight_layout()
add_conf_note(kind='filtered')
plt.show()

### Top 20 Genera by Abundance — All Confidences (COI)
Same marker but showing ALL genus assignments regardless of confidence, ranked by abundance. Bars are color-coded by mean confidence level.

In [ ]:
# Top 20 Genera by ABUNDANCE — All confidence levels
genus_col = f'{prefix_coi}_Genus'
conf_col = f'{prefix_coi}_Genus_Conf'
df_tmp = df_coi.copy()
df_tmp[genus_col] = df_tmp[genus_col].fillna('Unassigned')

genus_abund = df_tmp.groupby(genus_col)[sample_cols_coi].sum()
genus_abund = genus_abund.drop('Unassigned', errors='ignore')
genus_abund['Total'] = genus_abund.sum(axis=1)
genus_abund = genus_abund.sort_values('Total', ascending=False)
top20 = genus_abund.head(20)

# Mean confidence per genus
raw_conf = {}
if conf_col in df_tmp.columns:
    for genus in top20.index:
        mask = df_tmp[genus_col] == genus
        vals = pd.to_numeric(df_tmp.loc[mask, conf_col], errors='coerce').dropna()
        raw_conf[genus] = vals.mean() if len(vals) > 0 else None

fig, ax = plt.subplots(figsize=(11, 8))
colors = []
for genus in top20.index:
    c = raw_conf.get(genus)
    if c is None:
        colors.append('#999999')
    elif c >= 0.8:
        colors.append('#2d8a4e')
    elif c >= 0.5:
        colors.append('#e6a817')
    else:
        colors.append('#c0392b')

bars = ax.barh(range(len(top20)), top20['Total'].values[::-1], color=colors[::-1], height=0.7)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20.index[::-1], fontsize=9)
ax.set_xlabel('Total Abundance', fontsize=11)
ax.set_title(f'Top 20 Genera by Abundance — All Confidences (COI, {prefix_coi})',
             fontweight='bold', fontsize=12)

xmax = top20['Total'].max()
for j, genus in enumerate(top20.index[::-1]):
    c = raw_conf.get(genus)
    if c is not None:
        color = '#2d8a4e' if c >= 0.8 else '#e6a817' if c >= 0.5 else '#c0392b'
        ax.text(top20.loc[genus, 'Total'] + xmax * 0.01, j,
                f' {c:.2f}', va='center', fontsize=9, fontweight='bold', color=color)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2d8a4e', label='\u2265 0.80 (confident)'),
    Patch(facecolor='#e6a817', label='0.50 \u2013 0.79 (moderate)'),
    Patch(facecolor='#c0392b', label='< 0.50 (low confidence)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=8,
          title='Mean confidence', title_fontsize=9, framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
add_conf_note(kind='unfiltered')
plt.show()

## Forensics: BLAST Identification (JEDI + COI)
BLAST validation of the top OTUs for both markers provides ground-truth species identification beyond what SINTAX can offer with the PR2 database.

In [ ]:
def parse_blast_file(filepath):
    """Parses the custom text output from script 6_blast_top_otus.py"""
    data = []
    try:
        with open(filepath, 'r') as f:
            lines = f.readlines()
        start_reading = False
        for line in lines:
            if line.startswith('---'):
                start_reading = True
                continue
            if not start_reading or not line.strip(): continue
            parts = line.split('|')
            if len(parts) >= 4:
                otu_id = parts[0].strip()
                species = parts[2].strip()
                identity_str = parts[3].strip().replace('%', '')
                try:
                    reads = float(parts[1].strip())
                    identity = float(identity_str) if identity_str and identity_str != '-' else None
                    data.append({'OTU': otu_id, 'Species': species, 'Abundance': reads, 'Identity': identity})
                except: continue
    except FileNotFoundError:
        print("BLAST file not found.")
        return pd.DataFrame()
    return pd.DataFrame(data)

df_blast = parse_blast_file('out/Soil_eDNA_JEDI_COI_14_01_26/blast_results/blast_top10_JEDI.txt')
if not df_blast.empty:
    df_blast = df_blast.sort_values('Abundance', ascending=True)

    # Build clean labels: Species (short OTU suffix)
    labels = []
    for _, row in df_blast.iterrows():
        otu_short = row['OTU'].split('_')[-1]  # e.g. "005421"
        labels.append(f"{row['Species']}  [{otu_short}]")
    df_blast['Label'] = labels

    # Color by identity tier
    colors = []
    for _, row in df_blast.iterrows():
        if row['Identity'] is None:
            colors.append('#999999')
        elif row['Identity'] >= 97:
            colors.append('#2d8a4e')   # species-level green
        elif row['Identity'] >= 90:
            colors.append('#e6a817')   # genus-level amber
        else:
            colors.append('#c0392b')   # uncertain red

    fig, ax = plt.subplots(figsize=(11, max(3, len(df_blast) * 0.55)))
    bars = ax.barh(range(len(df_blast)), df_blast['Abundance'].values,
                   color=colors, edgecolor='white', height=0.7)

    ax.set_yticks(range(len(df_blast)))
    ax.set_yticklabels(df_blast['Label'].values, fontsize=9)
    ax.set_xlabel('Relative Abundance', fontsize=11)
    ax.set_title('Top JEDI OTUs — Closest Species Match (BLAST vs NCBI)', fontweight='bold', fontsize=13, pad=12)

    # Annotate identity at bar end
    xmax = df_blast['Abundance'].max()
    for j, (_, row) in enumerate(df_blast.iterrows()):
        if row['Identity'] is not None:
            ax.text(row['Abundance'] + xmax * 0.015, j,
                    f"{row['Identity']:.1f}%", va='center',
                    fontsize=9, fontweight='bold', color=colors[j])
        else:
            ax.text(row['Abundance'] + xmax * 0.015, j,
                    "N/A", va='center', fontsize=9, color='#999999')

    # Add legend for identity tiers
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#2d8a4e', label='\u2265 97% (species)'),
        Patch(facecolor='#e6a817', label='90-97% (genus)'),
        Patch(facecolor='#c0392b', label='< 90% (uncertain)'),
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=8,
              title='Identity level', title_fontsize=9, framealpha=0.9)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlim(0, xmax * 1.18)

    plt.tight_layout()
    add_conf_note(kind='blast')
    plt.show()
else:
    print("No valid BLAST data found to plot.")

df_blast = parse_blast_file('out/Soil_eDNA_JEDI_COI_14_01_26/blast_results/blast_top10_COI.txt')
if not df_blast.empty:
    df_blast = df_blast.sort_values('Abundance', ascending=True)

    # Build clean labels: Species (short OTU suffix)
    labels = []
    for _, row in df_blast.iterrows():
        otu_short = row['OTU'].split('_')[-1]  # e.g. "005421"
        labels.append(f"{row['Species']}  [{otu_short}]")
    df_blast['Label'] = labels

    # Color by identity tier
    colors = []
    for _, row in df_blast.iterrows():
        if row['Identity'] is None:
            colors.append('#999999')
        elif row['Identity'] >= 97:
            colors.append('#2d8a4e')   # species-level green
        elif row['Identity'] >= 90:
            colors.append('#e6a817')   # genus-level amber
        else:
            colors.append('#c0392b')   # uncertain red

    fig, ax = plt.subplots(figsize=(11, max(3, len(df_blast) * 0.55)))
    bars = ax.barh(range(len(df_blast)), df_blast['Abundance'].values,
                   color=colors, edgecolor='white', height=0.7)

    ax.set_yticks(range(len(df_blast)))
    ax.set_yticklabels(df_blast['Label'].values, fontsize=9)
    ax.set_xlabel('Relative Abundance', fontsize=11)
    ax.set_title('Top COI OTUs — Closest Species Match (BLAST vs NCBI)', fontweight='bold', fontsize=13, pad=12)

    # Annotate identity at bar end
    xmax = df_blast['Abundance'].max()
    for j, (_, row) in enumerate(df_blast.iterrows()):
        if row['Identity'] is not None:
            ax.text(row['Abundance'] + xmax * 0.015, j,
                    f"{row['Identity']:.1f}%", va='center',
                    fontsize=9, fontweight='bold', color=colors[j])
        else:
            ax.text(row['Abundance'] + xmax * 0.015, j,
                    "N/A", va='center', fontsize=9, color='#999999')

    # Add legend for identity tiers
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#2d8a4e', label='\u2265 97% (species)'),
        Patch(facecolor='#e6a817', label='90-97% (genus)'),
        Patch(facecolor='#c0392b', label='< 90% (uncertain)'),
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=8,
              title='Identity level', title_fontsize=9, framealpha=0.9)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlim(0, xmax * 1.18)

    plt.tight_layout()
    add_conf_note(kind='blast')
    plt.show()
else:
    print("No valid BLAST data found to plot.")